### Problem Statement 4D

4D. Building a Recommendation System with CBOW Word Embeddings
Problem: Use CBOW word embeddings to build a content-based recommendation system. For example, recommend similar articles or tweets based on the embeddings of keywords in a document.
Steps:
1. Data Collection and Preprocessing: Gather a dataset with textual content. Tokenize, clean, prepare for training.
2. Train CBOW Model:
a. Train the Word2Vec model using CBOW (sg=0) to learn word embeddings.
b. For a given article (or tweet), calculate the average of the embeddings of all the words in the document.
c. For each document in the dataset, calculate the cosine similarity between the document's average word embedding and the user's input embedding.
d. Document Representation: Represent each document as a vector by averaging the embeddings of all words in the document.
3. Recommendation Logic: Recommend the most similar documents using cosine similarity between their vector representations.

In [ ]:
# Optional installation:
# !pip install gensim scikit-learn

import os
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Step 1: Data Collection and Preprocessing
csv_path = "../LP IV Updated Datasets/CBOW/twitter_dataset.csv"

data = pd.read_csv(csv_path).head(1000)

# Tokenize and clean text
documents = []
for text in data['Text']:
    words = [w.lower() for w in str(text).split() if w.isalpha()]
    documents.append(words)

print(f"Loaded and tokenized {len(documents)} documents.")
print("Sample document:", documents[0])

In [ ]:
# Step 2: Train CBOW Model and represent documents as vectors
# Train Word2Vec using CBOW (sg=0)
cbow_model = Word2Vec(documents, vector_size=100, window=5, min_count=1, sg=0, epochs=10)

# Function to calculate average embedding of words in a document
def get_average_embedding(word_list):
    vectors = [cbow_model.wv[w] for w in word_list if w in cbow_model.wv]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

# Represent each document as a vector
doc_vectors = np.array([get_average_embedding(doc) for doc in documents])
print("Document matrix shape:", doc_vectors.shape)

In [ ]:
# Step 3: Recommendation Logic using Cosine Similarity
def recommend(query_text, top_k=3):
    print(f"\nUser Query: '{query_text}'")
    query_words = [w.lower() for w in query_text.split() if w.isalpha()]
    
    # Calculate user input embedding
    query_vector = get_average_embedding(query_words).reshape(1, -1)
    
    # Calculate cosine similarity with all documents in dataset
    scores = cosine_similarity(query_vector, doc_vectors).flatten()
    
    # Get top matching document indices
    top_indices = scores.argsort()[::-1][:top_k]
    
    print("Top Recommendations:")
    for rank, idx in enumerate(top_indices, 1):
        print(f"{rank}. [Score: {scores[idx]:.4f}] {data['Text'].iloc[idx]}")

# Test recommendations
recommend("happy sunny day with great friends")
recommend("new technology and software updates")